# 02 — Retention: one canonical chronological protocol
Updated with AI assistance; docs/AI_USE.md. Model logic lives in src/train.py.
`python -m src.train --trials 12` reruns the pipeline; this notebook reads its saved evidence.
It does not introduce a random split, tune on the test period or silently retrain.

In [ ]:
import json
import sys
from pathlib import Path
import pandas as pd
from IPython.display import display, Image
ROOT = Path.cwd()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
def report(name):
    return json.loads((ROOT / "reports/current_run" / name).read_text())

In [ ]:
from src.contracts import FEATURE_NAMES
metrics = report("retention_metrics.json")
metadata = json.loads((ROOT / "artifacts/retention_model_metadata.json").read_text())
assert metadata["feature_names"] == FEATURE_NAMES
print("Model", metadata["model_version"], "contract", metadata["contract_version"], "features", metadata["feature_count"])
display(pd.DataFrame(metrics["splits"]).T)
display(pd.DataFrame(metrics["model_comparison_validation"]))

## Genuine resource pruning
Eight classifier families and two baselines are compared before test evaluation. The top two
tunable families get 12 trials each. MedianPruner evaluates genuinely fitted earlier 50%, 75%
and 100% training-row subsets against the fixed later validation period. The final estimator
refits train plus validation. March fits calibrators and June selects calibration and threshold.

In [ ]:
trials = pd.DataFrame(metrics["optuna_trials"])
print(metrics["pruning"])
display(trials[["family", "trial", "state", "validation_pr_auc"]])
print("Pruned trials:", (trials["state"] == "pruned").sum())
display(pd.DataFrame({k:{"brier_score":v["brier_score"]} for k,v in metrics["calibration_decision_period"].items()}).T)
print("Selected calibration:", metrics["selected_calibration"])

In [ ]:
diag = report("retention_diagnostics.json")
display(pd.DataFrame({"model":diag["model"], "recency_same_count":diag["recency_same_selection_count"]}).T)
display(pd.DataFrame(diag["uncertainty"]["ci95"], index=["lower", "upper"]).T)
for name in ["precision_recall_curve.png", "roc_curve.png", "reliability_curve.png", "threshold_utility.png"]:
    display(Image(filename=str(ROOT / "reports/current_run/figures" / name)))

## Interpret the policy honestly
Offer utility is pM−C, no-offer utility is −pF and theoretical break-even is C/(M+F).
C=5, M=25, F=0 are scenarios: 20% theoretical, 20.5% selected on June outcomes for model 0.3.0.
The model predicts purchases, not incremental coupon response. Equal-count recency nearly matches
recall. The original 25-feature model is archived; September was reused after a feature-removal
decision based on June. Customer-bootstrap intervals do not create a new independent holdout.